<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Agent_memory_multi_agent_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
long_term_memory = {}

def save_memory(fact, config):
    # TODO: read the user ID from config["configurable"]
    user_id=config["configurable"]["user_id"]
    # TODO: read the thread ID from config["configurable"]
    thread_id=config["configurable"]["thread_id"]
    # TODO: get this user's fact list (empty list if none)
    facts=[]
    if user_id in long_term_memory:
        facts=long_term_memory[user_id]
    # TODO: append the fact only if it is new, then store the list back
    if fact not in facts:
        facts.append(fact)
        long_term_memory[user_id]=facts
    # TODO: return "Memory saved."
    return "Memory saved."

def build_system_prompt(user_id):
    facts = long_term_memory.get(user_id, [])
    lines = ["What you remember about this customer:"]
    lines += ["- " + f for f in facts]
    return "\n".join(lines)

monday = {"configurable": {"user_id": "cust_11", "thread_id": "monday"}}
friday = {"configurable": {"user_id": "cust_11", "thread_id": "friday"}}
other = {"configurable": {"user_id": "cust_42", "thread_id": "t1"}}

print(save_memory("Prefers short answers", monday))
print(save_memory("Prefers evening delivery", other))
print(save_memory("Prefers short answers", friday))
print(save_memory("Reads replies on a phone", friday))
print(long_term_memory)
print(build_system_prompt("cust_11"))

Memory saved.
Memory saved.
Memory saved.
Memory saved.
{'cust_11': ['Prefers short answers', 'Reads replies on a phone'], 'cust_42': ['Prefers evening delivery']}
What you remember about this customer:
- Prefers short answers
- Reads replies on a phone


In [4]:
monday["configurable"]['thread_id']

'monday'

In [ ]:
# Supervisor reply:
# - Order 201 (Desk lamp) is out for delivery, expected by 7 PM.
# - Order 202 (Wireless earbuds) was delivered 6 days ago, within the 10-day window, so a return is possible.
# - Order 203 (Office chair) was delivered 12 days ago, outside the 10-day window, so a return is not possible.

In [9]:
ORDERS = {
    "201": {"item": "Desk lamp", "status": "out for delivery", "eta": "7 PM", "delivered_days_ago": None},
    "202": {"item": "Wireless earbuds", "status": "delivered", "eta": None, "delivered_days_ago": 6},
    "203": {"item": "Office chair", "status": "delivered", "eta": None, "delivered_days_ago": 12},
}
POLICY = {"return_window_days": 10}

def get_order_details(order_id):
    return ORDERS[order_id]

def get_policy(topic):
    return POLICY[topic]

def ask_orders_team(order_id):
    o = get_order_details(order_id)
    return f"Order {order_id} ({o['item']}) is {o['status']}, expected by {o['eta']}."

def ask_refunds_team(order_id):
    # TODO: look up the order AND the return window
    o = get_order_details(order_id)
    w = get_policy("return_window_days")
    # TODO: build the "within" or "outside" sentence shown in Expected Output
    if o['delivered_days_ago'] <= w:
        return f"Order {order_id} ({o['item']}) was delivered {o['delivered_days_ago']} days ago, within the {w}-day window, so a return is possible."
    else:
      return f"Order {order_id} ({o['item']}) was delivered {o['delivered_days_ago']} days ago, outside of the {w}-day window, so a return is not possible"

TEAMS = {"orders": ask_orders_team, "refunds": ask_refunds_team}

def supervisor(parts):
    # TODO: send each (team, order_id) part to the matching team
    for part in parts:
        team, order_id = part
        answer = TEAMS[team](order_id)
        print(answer)
    # TODO: combine the answers into one reply, as shown in Expected Output

parts = [("orders", "201"), ("refunds", "202"), ("refunds", "203")]
print(supervisor(parts))

Order 201 (Desk lamp) is out for delivery, expected by 7 PM.
Order 202 (Wireless earbuds) was delivered 6 days ago, within the 10-day window, so a return is possible.
Order 203 (Office chair) was delivered 12 days ago, outside of the 10-day window, so a return is not possible
None


In [ ]:
# input: 96x96x3
# conv (16 filters, 5x5): 96x96x16
# pool: 48x48x16
# conv (32 filters, 3x3): 46x46x32
# pool: 23x23x32
# conv (64 filters, 3x3): 12x12x64

In [14]:
input_shape = (96, 96, 3)
layers = [
    {"type": "conv", "filters": 16, "size": 5, "padding": 2, "stride": 1},
    {"type": "pool"},
    {"type": "conv", "filters": 32, "size": 3, "padding": 0, "stride": 1},
    {"type": "pool"},
    {"type": "conv", "filters": 64, "size": 3, "padding": 1, "stride": 2},
]

def conv_out(n, f, p, s):
    # Return the output size for one dimension
    return (n - f + 2 * p ) // s + 1

def pool_out(n):
    # Return the output size for one dimension
    return n // 2

def trace_shapes(input_shape, layers):
    h, w, c = input_shape
    print(f"input: {h}x{w}x{c}")
    for layer in layers:
        if layer["type"] == "conv":
            h = conv_out(h, layer["size"], layer["padding"], layer["stride"])
            w = conv_out(w, layer["size"], layer["padding"], layer["stride"])
            c = layer["filters"]
            print(f"conv ({layer['filters']} filters, {layer['size']}x{layer['size']}): {h}x{w}x{c}")
        elif layer["type"] == "pool":
            h = pool_out(h)
            w = pool_out(w)
            print(f"pool: {h}x{w}x{c}")

trace_shapes(input_shape, layers)

input: 96x96x3
conv (16 filters, 5x5): 96x96x16
pool: 48x48x16
conv (32 filters, 3x3): 46x46x32
pool: 23x23x32
conv (64 filters, 3x3): 12x12x64
